# Phase 10B: Binary TabularGRPO for Fraud Classification

This notebook tests TabularGRPO in the form closest to the shared public implementation:

`Transaction features → Legitimate or Fraud`

It does **not** directly predict Approve, Verify, Review, or Decline. Those actions should be added only after a reliable fraud classifier exists.

## Experiment design

1. Use the chronological 60/20/20 split.
2. Train a supervised MoE Transformer warm-start.
3. Copy the warm-start model into a Binary GRPO classifier.
4. Guarantee both class actions, `0` and `1`, inside every GRPO group.
5. Monitor validation PR-AUC, probability separation, entropy, KL, zero-variance groups, and expert routing.
6. Select the best checkpoint using validation PR-AUC.
7. Select the classification threshold using validation F1.
8. Evaluate once on the untouched test partition.
9. Compare supervised warm-start versus Binary GRPO.
10. Repeat for multiple random seeds.

> Research POC only. Synthetic-data results do not establish real banking performance.


## Simple mental model

For one transaction, the model has only two possible answers:

- `0`: Legitimate
- `1`: Fraud

GRPO samples several answers, checks which answers match the known label, and gives better learning signal to correct answers. Unlike the earlier four-action experiment, the reward is direct and verifiable: correct prediction gets `1`, incorrect prediction gets `0`.


In [ ]:
# 1. Mount Google Drive and configure paths
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import copy, hashlib, json, os, platform, random, shutil, subprocess, sys, time

REPO='https://github.com/deepak-malviyaa/enterprise-adaptive-learning-lab.git'
PROJECT_ROOT=Path('/content/enterprise-adaptive-learning-lab')
DRIVE_ROOT=Path('/content/drive/MyDrive/enterprise-adaptive-learning-lab-data')
RUNS_ROOT=DRIVE_ROOT/'outputs'/'runs'; RUNS_ROOT.mkdir(parents=True,exist_ok=True)
USE_SYNTHETIC_DATA=True
DATASET_NAME='synthetic' if USE_SYNTHETIC_DATA else 'ulb_real'
DATA_PATH=(DRIVE_ROOT/'synthetic'/'synthetic_creditcard_compatible_100k.csv') if USE_SYNTHETIC_DATA else (DRIVE_ROOT/'raw'/'creditcard.csv')
print('Dataset:',DATA_PATH)


In [ ]:
# 2. Clone/update the project and install dependencies
if (PROJECT_ROOT/'.git').exists(): result=subprocess.run(['git','-C',str(PROJECT_ROOT),'pull','--ff-only'],text=True,capture_output=True)
elif PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT); result=subprocess.run(['git','clone',REPO,str(PROJECT_ROOT)],text=True,capture_output=True)
else: result=subprocess.run(['git','clone',REPO,str(PROJECT_ROOT)],text=True,capture_output=True)
print(result.stdout); print(result.stderr)
if result.returncode!=0: raise RuntimeError('Git operation failed.')
if not DATA_PATH.exists(): raise FileNotFoundError(DATA_PATH)
pyproject=PROJECT_ROOT/'pyproject.toml'; t=pyproject.read_text().replace('>=3.10,<3.13','>=3.10,<3.14'); pyproject.write_text(t)
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','torch'],check=True)


In [ ]:
# 3. Imports and reproducibility
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader,TensorDataset,WeightedRandomSampler
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score,roc_auc_score,precision_score,recall_score,f1_score,brier_score_loss,confusion_matrix
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEEDS=[42,123,456]
print('Device:',DEVICE,'Seeds:',SEEDS)


## Chronological data preparation

Training uses the earliest 60%, validation the next 20%, and test the latest 20%. Validation and test are never balanced artificially.


In [ ]:
# 4. Load, validate, split, and scale
def sha256(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
DATASET_SHA256=sha256(DATA_PATH); df=pd.read_csv(DATA_PATH)
EXPECTED=['Time']+[f'V{i}' for i in range(1,29)]+['Amount','Class']
missing=sorted(set(EXPECTED)-set(df.columns))
if missing or set(df.Class.dropna().unique())!={0,1}: raise ValueError(f'Invalid dataset: {missing}')
df=df.sort_values('Time',kind='mergesort').reset_index(drop=True); df['row_id']=np.arange(len(df))
a=int(len(df)*.60); b=int(len(df)*.80)
train_df=df.iloc[:a].copy(); val_df=df.iloc[a:b].copy(); test_df=df.iloc[b:].copy()
FEATURES=['Time']+[f'V{i}' for i in range(1,29)]+['Amount']
Xtr=train_df[FEATURES].copy(); Xv=val_df[FEATURES].copy(); Xt=test_df[FEATURES].copy()
ytr=train_df.Class.to_numpy(np.int64); yv=val_df.Class.to_numpy(np.int64); yt=test_df.Class.to_numpy(np.int64)
scaler=StandardScaler().fit(Xtr[['Time','Amount']])
for x in [Xtr,Xv,Xt]: x.loc[:,['Time','Amount']]=scaler.transform(x[['Time','Amount']])
Xtr=Xtr.to_numpy(np.float32); Xv=Xv.to_numpy(np.float32); Xt=Xt.to_numpy(np.float32)
VAL_X=torch.tensor(Xv,dtype=torch.float32,device=DEVICE); TEST_X=torch.tensor(Xt,dtype=torch.float32,device=DEVICE)
print('Rows:',len(df),'Frauds train/val/test:',ytr.sum(),yv.sum(),yt.sum(),'Checksum:',DATASET_SHA256)


## Balanced training without double weighting

The training loader samples fraud more often. The supervised loss is ordinary cross-entropy, so fraud is not weighted twice. Validation and test keep natural prevalence.


In [ ]:
# 5. Balanced training loader factory
train_dataset=TensorDataset(torch.tensor(Xtr),torch.tensor(ytr,dtype=torch.long))
class_counts=np.bincount(ytr); row_weights=(1.0/class_counts)[ytr]
def make_loader(seed):
    generator=torch.Generator(); generator.manual_seed(seed)
    sampler=WeightedRandomSampler(torch.tensor(row_weights,dtype=torch.double),num_samples=len(ytr),replacement=True,generator=generator)
    return DataLoader(train_dataset,batch_size=512,sampler=sampler)
print('Training class counts:',class_counts.tolist())


## MoE Transformer model

The model outputs two logits. Softmax converts them into legitimate and fraud probabilities. Expert-routing diagnostics are retained so GRPO cannot silently collapse the MoE.


In [ ]:
# 6. Model architecture
class Tokenizer(nn.Module):
    def __init__(self,n=30,d=48): super().__init__(); self.w=nn.Parameter(torch.randn(n,d)*.02); self.b=nn.Parameter(torch.zeros(n,d)); self.ids=nn.Parameter(torch.randn(n,d)*.02)
    def forward(self,x): return x.unsqueeze(-1)*self.w+self.b+self.ids
class MoE(nn.Module):
    def __init__(self,d=48,e=4):
        super().__init__(); self.router=nn.Linear(d,e); self.experts=nn.ModuleList([nn.Sequential(nn.Linear(d,96),nn.GELU(),nn.Linear(96,d)) for _ in range(e)]); self.last=None
    def forward(self,x):
        p=torch.softmax(self.router(x),-1); self.last=p; values,indices=p.topk(2,-1); out=torch.zeros_like(x)
        for k in range(2):
            for i,expert in enumerate(self.experts):
                mask=indices[:,k]==i
                if mask.any(): out[mask]+=values[mask,k:k+1]*expert(x[mask])
        return out
    def balance_loss(self):
        m=self.last.mean(0); return ((m-1/len(m))**2).mean()
class MoETransformer(nn.Module):
    def __init__(self,input_dim=30,num_classes=2):
        super().__init__(); d=48; self.tokenizer=Tokenizer(input_dim,d); self.cls=nn.Parameter(torch.zeros(1,1,d)); layer=nn.TransformerEncoderLayer(d,4,96,.1,batch_first=True,norm_first=True); self.encoder=nn.TransformerEncoder(layer,2,enable_nested_tensor=False); self.moe=MoE(d,4); self.head=nn.Linear(d,num_classes)
    def forward(self,x):
        z=self.tokenizer(x); z=self.encoder(torch.cat([self.cls.expand(len(x),-1,-1),z],1))[:,0]; z=z+self.moe(z); return self.head(z)


In [ ]:
# 7. Evaluation and threshold helpers
def evaluate_probabilities(labels,prob,threshold=.5):
    pred=(prob>=threshold).astype(int); tn,fp,fn,tp=confusion_matrix(labels,pred,labels=[0,1]).ravel()
    return {'pr_auc':float(average_precision_score(labels,prob)),'roc_auc':float(roc_auc_score(labels,prob)),'precision':float(precision_score(labels,pred,zero_division=0)),'recall':float(recall_score(labels,pred,zero_division=0)),'f1':float(f1_score(labels,pred,zero_division=0)),'brier':float(brier_score_loss(labels,prob)),'threshold':float(threshold),'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}
def infer(model,X):
    model.eval()
    with torch.no_grad(): logits=model(X); probs=torch.softmax(logits,-1); routing=model.moe.last.cpu().numpy()
    return probs[:,1].cpu().numpy(), routing

def choose_threshold(labels,prob):
    rows=[evaluate_probabilities(labels,prob,float(t)) for t in np.arange(.01,1,.01)]
    table=pd.DataFrame(rows); chosen=table.sort_values(['f1','recall'],ascending=[False,False]).iloc[0]
    return float(chosen.threshold),table


## Stage 1: supervised warm-start

The model first learns ordinary fraud classification using verified labels. The best checkpoint is selected by validation PR-AUC.


In [ ]:
# 8. Supervised training function
def supervised_warmstart(seed,epochs=25,patience=5):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); loader=make_loader(seed); model=MoETransformer().to(DEVICE); optimizer=torch.optim.AdamW(model.parameters(),lr=5e-4,weight_decay=1e-4); loss_fn=nn.CrossEntropyLoss(); best=-1.; state=None; stale=0; history=[]
    for epoch in range(1,epochs+1):
        model.train(); total=0.
        for xb,yb in loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); optimizer.zero_grad(set_to_none=True); logits=model(xb); loss=loss_fn(logits,yb)+.1*model.moe.balance_loss(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),5.); optimizer.step(); total+=loss.item()*len(xb)
        vp,routing=infer(model,VAL_X); pr=average_precision_score(yv,vp); entropy=float((-routing*np.log(routing+1e-9)).sum(1).mean()); history.append({'epoch':epoch,'loss':total/len(train_dataset),'val_pr_auc':float(pr),'routing_entropy':entropy})
        print('supervised',seed,epoch,'PR',round(pr,4),'entropy',round(entropy,4))
        if pr>best: best=pr; state=copy.deepcopy(model.state_dict()); stale=0
        else: stale+=1
        if stale>=patience: break
    model.load_state_dict(state); return model,pd.DataFrame(history)


## Stage 2: Binary GRPO fine-tuning

Every group contains at least one `0` and one `1`. Correct class predictions receive reward `1`; incorrect predictions receive `0`. The old policy supplies the denominator for clipped policy ratios. Validation PR-AUC, not precision alone, selects the checkpoint.


In [ ]:
# 9. Binary GRPO fine-tuning
GROUP_SIZE=10; CLIP=.2; KL_COEF=.01; ENTROPY_COEF=.01; BALANCE_COEF=.1

def grpo_finetune(warm_model,seed,epochs=12,patience=4):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); loader=make_loader(seed); model=copy.deepcopy(warm_model).to(DEVICE); optimizer=torch.optim.AdamW(model.parameters(),lr=1e-4,weight_decay=1e-5); best=-1.; state=None; stale=0; history=[]
    for epoch in range(1,epochs+1):
        old_model=copy.deepcopy(model).eval(); model.train(); total=0.; zero_groups=0; seen=0; last_kl=last_entropy=0.
        for xb,yb in loader:
            xb,yb=xb.to(DEVICE),yb.to(DEVICE); optimizer.zero_grad(set_to_none=True)
            with torch.no_grad():
                old_logits=old_model(xb); old_probs=torch.softmax(old_logits,-1)
                fixed=torch.tensor([0,1],device=DEVICE).unsqueeze(0).expand(len(xb),-1)
                sampled=torch.multinomial(old_probs,GROUP_SIZE-2,replacement=True)
                actions=torch.cat([fixed,sampled],1)
                old_selected=old_probs.gather(1,actions)
                rewards=(actions==yb.unsqueeze(1)).float(); std=rewards.std(1,keepdim=True); advantages=(rewards-rewards.mean(1,keepdim=True))/(std+1e-8); zero_groups+=(std<1e-6).sum().item(); seen+=len(xb)
            current_probs=torch.softmax(model(xb),-1); current_selected=current_probs.gather(1,actions); ratio=current_selected/(old_selected+1e-8); unclipped=ratio*advantages; clipped=torch.clamp(ratio,1-CLIP,1+CLIP)*advantages; surrogate=torch.where(advantages>=0,torch.minimum(unclipped,clipped),torch.maximum(unclipped,clipped)); policy_loss=-surrogate.mean()
            old_safe=old_probs.clamp_min(1e-8); current_safe=current_probs.clamp_min(1e-8); kl=(old_safe*(old_safe.log()-current_safe.log())).sum(1).mean(); entropy=-(current_safe*current_safe.log()).sum(1).mean(); loss=policy_loss+KL_COEF*kl-ENTROPY_COEF*entropy+BALANCE_COEF*model.moe.balance_loss(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),5.); optimizer.step(); total+=loss.item(); last_kl=float(kl.item()); last_entropy=float(entropy.item())
        vp,routing=infer(model,VAL_X); pr=average_precision_score(yv,vp); route_entropy=float((-routing*np.log(routing+1e-9)).sum(1).mean()); history.append({'epoch':epoch,'loss':total/len(loader),'val_pr_auc':float(pr),'zero_variance_fraction':zero_groups/seen,'kl':last_kl,'policy_entropy':last_entropy,'routing_entropy':route_entropy})
        print('grpo',seed,epoch,'PR',round(pr,4),'zero',round(zero_groups/seen,4),'policy H',round(last_entropy,4),'routing H',round(route_entropy,4))
        if pr>best: best=pr; state=copy.deepcopy(model.state_dict()); stale=0
        else: stale+=1
        if stale>=patience: break
    model.load_state_dict(state); return model,pd.DataFrame(history)


## Run multiple seeds

Multiple seeds reveal whether an improvement is stable or happened only once by chance.


In [ ]:
# 10. Run supervised and GRPO experiments for every seed
runs=[]
for seed in SEEDS:
    supervised_model,supervised_history=supervised_warmstart(seed)
    supervised_val,_=infer(supervised_model,VAL_X); supervised_threshold,_=choose_threshold(yv,supervised_val); supervised_test,_=infer(supervised_model,TEST_X); supervised_metrics=evaluate_probabilities(yt,supervised_test,supervised_threshold)
    grpo_model,grpo_history=grpo_finetune(supervised_model,seed)
    grpo_val,grpo_routing=infer(grpo_model,VAL_X); grpo_threshold,grpo_thresholds=choose_threshold(yv,grpo_val); grpo_test,test_routing=infer(grpo_model,TEST_X); grpo_metrics=evaluate_probabilities(yt,grpo_test,grpo_threshold)
    runs.append({'seed':seed,'supervised_model':supervised_model,'supervised_history':supervised_history,'supervised_metrics':supervised_metrics,'grpo_model':grpo_model,'grpo_history':grpo_history,'grpo_metrics':grpo_metrics,'grpo_thresholds':grpo_thresholds,'test_routing':test_routing})
    print('seed',seed,'supervised',supervised_metrics,'grpo',grpo_metrics)


In [ ]:
# 11. Compare supervised versus GRPO across seeds
rows=[]
for run in runs:
    rows.append({'seed':run['seed'],'stage':'supervised',**run['supervised_metrics']})
    rows.append({'seed':run['seed'],'stage':'binary_grpo',**run['grpo_metrics']})
results=pd.DataFrame(rows); display(results)
summary=results.groupby('stage').agg({'pr_auc':['mean','std'],'roc_auc':['mean','std'],'precision':['mean','std'],'recall':['mean','std'],'f1':['mean','std'],'fp':['mean','std'],'fn':['mean','std']})
display(summary)


## Final diagnostics and selection

Binary GRPO is considered useful only if its mean PR-AUC or F1 improves without large instability, expert collapse, or unacceptable false positives.


In [ ]:
# 12. Select the best binary-GRPO seed and inspect expert routing
best_run=max(runs,key=lambda r:r['grpo_metrics']['pr_auc']); BEST_MODEL=best_run['grpo_model']; BEST_SEED=best_run['seed']; BEST_METRICS=best_run['grpo_metrics']; routing=best_run['test_routing']; utilization=routing.mean(0); routing_entropy=float((-routing*np.log(routing+1e-9)).sum(1).mean())
print('Best seed:',BEST_SEED); print('Metrics:',json.dumps(BEST_METRICS,indent=2)); print('Expert utilization:',utilization); print('Routing entropy:',routing_entropy)


In [ ]:
# 13. Save complete experiment
RUN_ID=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+f'_binary_tabulargrpo_{DATASET_NAME}'; RUN_DIR=RUNS_ROOT/RUN_ID; RUN_DIR.mkdir(parents=True)
for run in runs:
    seed=run['seed']; torch.save(run['supervised_model'].state_dict(),RUN_DIR/f'supervised_seed_{seed}.pt'); torch.save(run['grpo_model'].state_dict(),RUN_DIR/f'binary_grpo_seed_{seed}.pt'); run['supervised_history'].to_csv(RUN_DIR/f'supervised_history_seed_{seed}.csv',index=False); run['grpo_history'].to_csv(RUN_DIR/f'grpo_history_seed_{seed}.csv',index=False); run['grpo_thresholds'].to_csv(RUN_DIR/f'grpo_thresholds_seed_{seed}.csv',index=False)
results.to_csv(RUN_DIR/'seed_results.csv',index=False); summary.to_csv(RUN_DIR/'summary.csv');
config={'dataset_name':DATASET_NAME,'dataset_sha256':DATASET_SHA256,'group_size':GROUP_SIZE,'clip_epsilon':CLIP,'kl_coef':KL_COEF,'entropy_coef':ENTROPY_COEF,'balance_coef':BALANCE_COEF,'seeds':SEEDS,'best_seed':BEST_SEED,'best_metrics':BEST_METRICS,'expert_utilization':utilization.tolist(),'routing_entropy':routing_entropy,'warning':'Research POC only'}
(RUN_DIR/'config.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
print('Saved:',RUN_DIR)


## How to interpret the result

- If Binary GRPO improves PR-AUC/F1 over the same supervised model across most seeds, it adds value.
- If it improves only one seed, the result is unstable.
- If expert routing entropy collapses near zero, GRPO damaged the MoE.
- If supervised performance is better, keep supervised learning and do not use GRPO for this dataset.
